# TTD Databricks SDK - Example Notebook

This notebook demonstrates how to use the `ttd-databricks` SDK to submit audience data
to The Trade Desk's Data API from a Databricks environment.

## Prerequisites
- Package installed: `%pip install ttd-databricks`
- A valid TTD API token (TTD-Auth)
- A valid TTD Data Provider ID
- A valid TTD Advertiser ID (for advertiser endpoint)

In [ ]:
%pip install ttd-databricks

# Recommended to restart kerner to use updated packages
dbutils.library.restartPython()

## Step 1: Configure credentials

In production, retrieve secrets from Databricks Secrets:
```python
api_token = dbutils.secrets.get(scope="ttd", key="api-token")
```

In [ ]:
# Replace with your actual values
API_TOKEN = "<your-ttd-auth-token>"
DATA_PROVIDER_ID = "<your-data-provider-id>"
ADVERTISER_ID = "<your-advertiser-id>"

## Step 2: Create the client

Use `from_params()` to create the client from your credentials.
The SparkSession is auto-detected from the Databricks runtime.

In [ ]:
from ttd_databricks_python.ttd_databricks import (
    TtdDatabricksClient,
    AdvertiserContext,
    TTDEndpoint,
    get_ttd_input_schema,
    TTDSchemaValidationError,
    TTDApiError,
)

client = TtdDatabricksClient.from_params(api_token=API_TOKEN)
print("Client ready.")

## Step 3: Create a context

The context holds endpoint-specific config: which advertiser and data provider
the data belongs to.

In [ ]:
context = AdvertiserContext(
    data_provider_id=DATA_PROVIDER_ID,
    advertiser_id=ADVERTISER_ID,
)
print(f"Context: {context}")

## Step 4: Inspect the required input schema

Use `get_ttd_input_schema()` to see which columns your DataFrame must contain.

For more information on the meaning of particular fields and supported data types supported per endpoint refer to the following table:

| Endpoint | Context | Data API | Documentation |
|---|---|---|---|
| Advertiser | `AdvertiserContext` | `POST /data/advertiser` | [OpenTTD](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/audience/doc/post-data-advertiser-external) |
| Third Party | `ThirdPartyContext` | `POST /data/thirdparty` | [OpenTTD](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/audience/doc/post-data-thirdparty) |
| Offline Conversion | `OfflineConversionContext` | `POST /providerapi/offlineconversion` | [OpenTTD](https://open.thetradedesk.com/advertiser/docsApp/GuidesAdvertiser/data/doc/post-providerapi-offlineconversion) |
| Deletion / Opt-Out — Advertiser | `DeletionOptOutAdvertiserContext` | `POST /data/deletion-optout/advertiser` | [OpenTTD](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/audience/doc/post-data-deletion-optout-advertiser-external) |
| Deletion / Opt-Out — Third Party | `DeletionOptOutThirdPartyContext` | `POST /data/deletion-optout/thirdparty` | [OpenTTD](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/audience/doc/post-data-deletion-optout-thirdparty) |
| Deletion / Opt-Out — Merchant | `DeletionOptOutMerchantContext` | `POST /data/deletion-optout/merchant` | [OpenTTD](https://open.thetradedesk.com/provider/docsApp/GuidesProvider/retail/doc/post-data-deletion-optout-merchant) |

In [ ]:
input_schema = get_ttd_input_schema(TTDEndpoint.ADVERTISER)
print("Required input schema:")

for field in input_schema.fields:
  print(f" {field.name}: {field.dataType.simpleString()} (nullable={field.nullable})")

## Step 5: Prepare your input DataFrame

Your DataFrame must contain at minimum the mandatory columns.
Extra columns are allowed and will be preserved in the output.

In [ ]:
# Example: create a small sample DataFrame
# In practice, read from a Delta table or other data source
sample_data = [
      {"id_type": "tdid", "id_value": "a3f1c2d4-8e7b-4f6a-9c0d-1b2e3f4a5b6c", "segment_name": "segment_1"},
      {"id_type": "daid", "id_value": "7d9e0f1a-2b3c-4d5e-6f7a-8b9c0d1e2f3a", "segment_name": "segment_2"},
      # intentionally incorrect format for ramp_id to showcase error enrties in output
      {"id_type": "ramp_id", "id_value": "c4d5e6f7-a8b9-4c0d-1e2f-3a4b5c6d7e8f", "segment_name": "segment_3"},
      {"id_type": "tdid", "id_value": "1f2a3b4c-5d6e-4f7a-8b9c-0d1e2f3a4b5c", "segment_name": "segment_4"},
      {"id_type": "daid", "id_value": "9b0c1d2e-3f4a-4b5c-6d7e-8f9a0b1c2d3e", "segment_name": "segment_5"},
]

input_df = spark.createDataFrame(sample_data)
display(input_df)

## Step 6: Submit data via push_data (ad hoc mode)

`push_data` will:
1. Validate your DataFrame has the required columns
2. Send data in batches (default: 1600 rows per request)
3. Return a new DataFrame with all original columns plus status columns:
   `success`, `error_code`, `error_message`, `processed_timestamp`

In [ ]:
try:
    result_df = client.push_data(
        df=input_df,
        context=context,
        batch_size=1600,     # Number of rows batched together in a single request to The Trade Desk
    )
    display(result_df)
except TTDSchemaValidationError as e:
    print(f"Schema validation failed: {e}")
except TTDApiError as e:
    print(f"API call failed (HTTP {e.status_code}): {e}")

## Step 7: Inspect results

In [ ]:
from pyspark.sql.functions import col

total = result_df.count()
succeeded = result_df.filter(col("success") == True).count()
failed = total - succeeded

print(f"Total rows:  {total}")
print(f"Succeeded:   {succeeded}")
print(f"Failed:      {failed}")

if failed > 0:
    print("\nFailed rows:")
    display(result_df.filter(col("success") == False))

## Optional: Table setup for batch processing mode

Use the table setup utilities to create Delta tables with the correct schema,
then use `batch_process()` for ongoing incremental processing.

In [ ]:
# Creates three managed Delta tables in the active catalog/database.
# Default names: ttd_advertiser_input, ttd_advertiser_output, ttd_metadata
# Pass table_name= and location= to use custom names or external storage.
input_table = client.setup_input_table(endpoint=TTDEndpoint.ADVERTISER)
output_table = client.setup_output_table(endpoint=TTDEndpoint.ADVERTISER)
metadata_table = client.setup_metadata_table(table_name="ttd_advertiser_metadata")

print(f"Input table:    {input_table}")
print(f"Output table:   {output_table}")
print(f"Metadata table: {metadata_table}")

In [ ]:
from pyspark.sql import functions as F

# This cell simulates your upstream pipeline writing records to the input table
# The user is responsible to write into the input table, the SDK only performs reads from the table


# updated_at is required for incremental processing: batch_process uses it
# to filter rows added since the last run when process_new_records_only=True
# The user is responsible to set the updated_at value for entries in the input table

(
    spark.createDataFrame(sample_data)
          .withColumn("updated_at", F.current_timestamp())
          .write.format("delta")
          .mode("append")
          .saveAsTable(input_table)
)

display(spark.table(input_table))

In [ ]:
# Run batch processing (reads from input_table, writes to output_table)

# process_new_records_only=True filters to rows where updated_at > last run date
# On the first run, metadata_table is empty so all rows are processed
client.batch_process(
    context=context,
    input_table=input_table,
    output_table=output_table,
    metadata_table=metadata_table,
    process_new_records_only=True,   # Processes rows updated after last run; processes all rows on first run
    batch_size=1600,                 # Number of rows grouped together in a single request to The Trade Desk
    parallelism=16,                  # Number of paralellel workers processing the entries from the input table
)

In [ ]:
# Display the output table
display(spark.table(output_table))

# Display the metadata table
display(spark.table(metadata_table))